In [5]:
# Regression Diagnostics

This notebook evaluates the robustness of the main Ordinary Least Squares (OLS) regression results by identifying influential observations.

Rather than estimating new models, the notebook examines whether the estimated regression coefficients are disproportionately driven by a small number of Members of the European Parliament (MEPs).

The notebook implements standard regression diagnostics commonly reported in applied economics and political science.

## Workflow

1. Load the cleaned analysis dataset.
2. Recreate the main OLS regression model.
3. Compute DFBETA statistics.
4. Compute Cook's Distance.
5. Compute leverage (hat values).
6. Compute studentized residuals.
7. Produce influence plots.
8. Summarize influential observations.

The outputs generated here are intended as robustness checks for the main analysis rather than as alternative empirical models.

SyntaxError: unterminated string literal (detected at line 14) (2639234637.py, line 14)

In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import (
    OLSInfluence,
)

# ----------------------------------------------------------
# Project path setup
# ----------------------------------------------------------

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.config import PROJECT_ROOT, ensure_default_directories

ensure_default_directories()

DIAGNOSTICS_DIR = PROJECT_ROOT / "8 Regression Diagnostics"

FINAL_ANALYSIS_DIR = PROJECT_ROOT / "7 Final Analysis"

OUTPUTS_DIR = DIAGNOSTICS_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ANALYSIS_DATA_PATH = (
    FINAL_ANALYSIS_DIR
    / "outputs"
    / "tables"
    / "polarisation_with_demography.xlsx"
)

assert ANALYSIS_DATA_PATH.exists(), (
    f"Analysis dataset not found:\n{ANALYSIS_DATA_PATH}"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Analysis dataset: {ANALYSIS_DATA_PATH}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Tables directory: {TABLES_DIR}")

Project root: /Users/smrut/Desktop/Smruthi/Milan/Bocconi/coursework/Sem3/Population/Project/Population-1/Analysis Population Dynamics
Analysis dataset: /Users/smrut/Desktop/Smruthi/Milan/Bocconi/coursework/Sem3/Population/Project/Population-1/Analysis Population Dynamics/7 Final Analysis/outputs/tables/polarisation_with_demography.xlsx
Figures directory: /Users/smrut/Desktop/Smruthi/Milan/Bocconi/coursework/Sem3/Population/Project/Population-1/Analysis Population Dynamics/8 Regression Diagnostics/outputs/figures
Tables directory: /Users/smrut/Desktop/Smruthi/Milan/Bocconi/coursework/Sem3/Population/Project/Population-1/Analysis Population Dynamics/8 Regression Diagnostics/outputs/tables


In [2]:
## Load Cleaned Analysis Dataset

This section loads the cleaned dataset produced by the final analysis notebook.

The dataset contains:

- LDA topic shares
- LLM-generated polarisation scores
- Extracted MEP characteristics
- Country-level demographic variables

The loaded dataframe (`analysis_df`) will be used throughout the diagnostics notebook.

SyntaxError: invalid syntax (1036114044.py, line 3)

In [3]:
# ----------------------------------------------------------
# Load cleaned analysis dataset
# ----------------------------------------------------------

analysis_df = pd.read_excel(ANALYSIS_DATA_PATH)

print(f"Rows: {len(analysis_df):,}")
print(f"Columns: {len(analysis_df.columns)}")

display(analysis_df.head())

Rows: 193
Columns: 103


,mep_name_clean,mep_name_raw,n_documents,Ageing & Pensions,Borderline: Economic governance and employment,Borderline: Island regions and cohesion,Borderline: Labour market and social policy,Borderline: Regional development policy,Borderline: Youth / demographic future (weak signal),"Fertility, Families & Gender",...,dominant_topic,professional_background_clean,international_experience_flag,migration_background_flag,military_background_flag,parental_status_known,has_children_if_known,religion_mentioned_flag,education_field_clean,topic_polarisation_cluster
0,abir al sahlani,Abir Al-Sahlani,2,0.010289,0.000082,0.000082,0.000082,0.111688,0.031171,0.000246,...,Migration & Population Change,Politics / Public Administration,No,Yes,No,Yes,Yes,No,Information engineering,0
1,adam kosa,Ádám Kósa,3,0.200596,0.000285,0.000285,0.028850,0.000285,0.000285,0.348529,...,Other,Law,No,No,No,No,Unknown,No,Law,0
2,agnes le brun,Agnès Le Brun,1,0.001075,0.000215,0.000215,0.000215,0.391120,0.000215,0.000645,...,Other,Academia / Education,No,No,No,No,Unknown,No,Humanities / Education / Health,0
3,agnieszka koz owska rajewicz,Agnieszka Kozłowska-Rajewicz,1,0.000199,0.000040,0.000040,0.000040,0.102693,0.000040,0.830436,...,"Fertility, Families & Gender",Academia / Education,No,No,No,No,Unknown,No,STEM,0
4,alain cadec,Alain Cadec,1,0.542029,0.000339,0.000339,0.000339,0.326859,0.000339,0.001017,...,Ageing & Pensions,Politics / Public Administration,No,No,No,Yes,Yes,No,Economics / Business,0


In [6]:
analysis_df.columns.tolist()

['mep_name_clean',
 'mep_name_raw',
 'n_documents',
 'Ageing & Pensions',
 'Borderline: Economic governance and employment',
 'Borderline: Island regions and cohesion',
 'Borderline: Labour market and social policy',
 'Borderline: Regional development policy',
 'Borderline: Youth / demographic future (weak signal)',
 'Fertility, Families & Gender',
 'Health & Care Services',
 'Housing & Living Conditions',
 'Migration & Population Change',
 'Population Growth & Sustainability',
 'Regional Depopulation & Territorial Cohesion',
 'Youth & Future Generations',
 'Other',
 'dominant_cluster',
 'dominant_cluster_score',
 'n_relevant_speeches',
 'model_used',
 'n_relevant_speeches_cached',
 'dominant_cluster_cached',
 'polarisation_score',
 'confidence',
 'us_vs_them',
 'emotional_intensity',
 'moral_absolutism',
 'hostility_to_opponents',
 'summary',
 'evidence_quotes',
 'mep_name_old_from_extracted',
 'name',
 'wikipedia_language',
 'page_title',
 'page_url',
 'wikipedia_content_html',
 'wik

In [7]:
# ----------------------------------------------------------
# Verify required variables
# ----------------------------------------------------------

required_columns = [

    "polarisation_score",

    "ep_group_clean",

    "gender_bucket",

    "education_level_bucket",

    "professional_background_clean",

]

missing = [c for c in required_columns if c not in analysis_df.columns]

if missing:
    raise ValueError(
        "The following required variables are missing:\n"
        + "\n".join(missing)
    )

print("All required variables found.")

All required variables found.
